# Module 3: Generative AI & Retrieval-Augmented Generation (RAG)
## AI Study Assistant - From Scratch

Welcome to **Module 3**! Is notebook mein hum ek production-grade **RAG (Retrieval-Augmented Generation)** pipeline samjhenge aur build karenge jo student ke uploaded study notes (PDF ya TXT) se accurate, grounded answers provide karti hai bina kisi hallucination ke.

---

### Core RAG Concepts in Simple Hinglish:

1. **RAG Kyun Zaroori Hai? (Why RAG?)**:
   - General LLMs ko aapke specific class notes, syllabus, ya private PDFs ka access nahi hota.
   - Agar model ko bina context ke poochhein, toh wo *hallucinate* (mann-ghadant jhooth bolna) kar sakta hai.
   - RAG pehle relevant text retrieve karta hai aur fir usi text ke base par answer generate karta hai.

2. **Document Ingestion & Parsing**:
   - PDF ya TXT file se plain text extract karna along with metadata (document name, page number).

3. **Text Chunking with Overlap**:
   - Ek poori 50-page ki book model me ek sath nahi daali ja sakti.
   - Text ko 400 characters ke segments me split kiya jata hai with 60 characters ka **overlap**.
   - *Overlap kyun zaroori hai?* Taaki agar koi concept do chunks ke border par split ho raha ho, toh uska context break na ho.

4. **Vector Database (FAISS - Facebook AI Similarity Search)**:
   - Har text chunk ko Transformer (`all-MiniLM-L6-v2`) 384-dimensional dense vector me convert karta hai.
   - FAISS in millions of vectors ko RAM me index karta hai aur Cosine Similarity (Inner Product on L2-normalized vectors) ke zariye microsecond me closest matches dhoondh nikalta hai.

5. **Semantic Retrieval**:
   - Student ke question ko embed karke FAISS me search kiya jata hai.
   - Top-K most relevant chunks retrieve hote hain.

6. **Groundedness & Anti-Hallucination Guardrail**:
   - Agar similarity score threshold (0.35) se kam ho, toh system clearly bol deta hai:
     *"The uploaded study notes do not contain information regarding this topic."*
   - Model apni taraf se fabricated answer nahi banata aur hamesha page number aur source document cite karta hai.

## Step 1: Document Loading and Chunking

In [ ]:
import sys
sys.path.insert(0, "..")

from src.rag.document_loader import DocumentLoader
from src.rag.chunker import TextChunker

notes_path = "../data/raw/sample_study_notes.txt"
pages = DocumentLoader.load_document(notes_path)
print(f"Loaded {len(pages)} page(s) from {pages[0].document_name}")

chunker = TextChunker(chunk_size=400, chunk_overlap=60)
chunks = chunker.chunk_text(pages[0].text, document_name=pages[0].document_name, page_number=pages[0].page_number)
print(f"Created {len(chunks)} overlapping chunks.")
print("\nSample Chunk 0 Preview:")
print("Chunk ID:", chunks[0].chunk_id)
print("Text:\n", chunks[0].text)

## Step 2: Vector Embedding & FAISS Indexing

In [ ]:
from src.rag.pipeline import RAGPipeline

pipeline = RAGPipeline(
    index_path="../models/faiss_index.bin",
    metadata_path="../models/faiss_chunks.json"
)

stats = pipeline.index_document(notes_path)
print("Indexing Complete! Stats:", stats)

## Step 3: Grounded Q&A with Citations
Test a question that exists in the study notes.

In [ ]:
query = "How does L1 regularization differ from L2 in preventing overfitting?"
response = pipeline.query(query)

print("Question :", response["question"])
print("Found?   :", response["found_in_notes"])
print("Confidence:", response["confidence"])
print("\nAnswer:\n", response["answer"])
print("\nCitations:")
for c in response["citations"]:
    print(f"  Document: {c['document_name']} | Page: {c['page_number']} | Relevance: {c['relevance_score']}")
    print(f"  Snippet: {c['snippet']}\n")

## Step 4: Strict Anti-Hallucination Guardrail Test
Agar student koi aisi cheez poochhe jo notes me nahi hai, model invent nahi karega.

In [ ]:
out_of_scope_query = "What is the capital of Australia and who is the prime minister?"
response_ooc = pipeline.query(out_of_scope_query)

print("Question:", response_ooc["question"])
print("Found?  :", response_ooc["found_in_notes"])
print("Answer  :", response_ooc["answer"])
print("Citations Count:", len(response_ooc["citations"]))